# MLLM Teacher: Qwen2.5-Omni-3B on MIntRec 2.0

**Goal**: Download MIntRec 2.0 (text + audio + video, raw `.mp4` clips), explore the task, then run the frozen Qwen2.5-Omni-3B teacher over **video + audio + text** and verify hidden-state extraction.

This mirrors `src/early_experiment/mllm_teacher.ipynb` (FSC, audio-only) but uses the full tri-modal teacher input. Designed to run end-to-end on **RunPod** (Linux GPU); the 4-bit cell also fits a local 6 GB 3060 for a quick smoke test.

- Dataset: [thuiar/MIntRec2.0](https://github.com/thuiar/MIntRec2.0) (ICLR 2024) — 30 fine-grained intents, 9.3K in-scope + 5.7K out-of-scope utterances from *Superstore / Big Bang Theory / Friends*.
- Sample index convention: `dia{dialogue_id}_utt{utterance_id}`.

## 1. Setup & Installs

`gdown` pulls the Google-Drive folder; `qwen-omni-utils` handles video+audio packing for the Omni processor. The `[decord]` extra gives fast frame decoding. Safe to re-run — pip skips already-satisfied packages.

In [ ]:
# On a fresh RunPod image you may also need the HF stack + bitsandbytes; uncomment as needed.
%pip install -q gdown "qwen-omni-utils[decord]" librosa soundfile
# %pip install -q "transformers>=4.47.0" accelerate bitsandbytes av

In [1]:
import os
import sys

# Register FFmpeg shared DLLs on Windows so torchcodec/av can find avcodec/avformat.
# The return value MUST be stored — if GC'd, the dir is removed from the DLL search path.
# On Linux (RunPod) this whole block is skipped; ffmpeg comes from the system.
if sys.platform == 'win32':
    _ffmpeg_dll_dir = None
    for _p in os.environ.get('PATH', '').split(';'):
        if _p and os.path.exists(os.path.join(_p, 'avcodec-62.dll')):
            _ffmpeg_dll_dir = os.add_dll_directory(_p)
            break
    if _ffmpeg_dll_dir is None:
        print('WARNING: avcodec-62.dll not found on PATH — video/audio decoding may fail')
    else:
        print(f'FFmpeg DLL dir registered: {_p}')

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Audio as IPyAudio, display

# Put repo src/ on path so `import common.config` resolves regardless of launch dir.
# This notebook lives at src/mintrec/early_experiment/ -> parents[2] == src/.
_SRC = Path.cwd()
while _SRC.name != 'src' and _SRC != _SRC.parent:
    _SRC = _SRC.parent
if _SRC.name == 'src' and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))
from common.config import MINTREC_DATA, MINTREC_OUTPUTS

MINTREC_DATA.mkdir(parents=True, exist_ok=True)
MINTREC_OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f'PyTorch     : {torch.__version__}')
print(f'Data root   : {MINTREC_DATA}')
print(f'Output root : {MINTREC_OUTPUTS}')

FFmpeg DLL dir registered: C:\Users\lenovo\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg.Shared_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1.1-full_build-shared\bin
PyTorch     : 2.9.1+cu126
Data root   : D:\msc_AI\individual_project\multimodal-distillation-for-extreme-edge\data\mintrec
Output root : D:\msc_AI\individual_project\multimodal-distillation-for-extreme-edge\outputs\mintrec


In [2]:
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA : True
GPU  : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM : 6.4 GB


## 2. Download MIntRec 2.0

Pulls the whole Google-Drive folder (in-scope 13 GB + out-of-scope 7.44 GB → ~20 GB; allow time + disk). The folder holds a handful of large zips, so `gdown --folder` stays under its 50-file limit. Idempotent: skipped if the in-scope split is already present.

> If gdown rate-limits the folder, grab the folder manually and drop the zips into `MINTREC_DATA`, then re-run the unzip cell. Folder: https://drive.google.com/drive/folders/1W-z8kMOA1TaB3pE4rk4ZpkBQLvODMQ_Y

In [ ]:
import zipfile
import gdown

GDRIVE_FOLDER = 'https://drive.google.com/drive/folders/1W-z8kMOA1TaB3pE4rk4ZpkBQLvODMQ_Y'

# Heuristic 'already downloaded' check: any train.tsv anywhere under MINTREC_DATA.
already = list(MINTREC_DATA.rglob('train.tsv'))
if already:
    print(f'Found existing annotations ({len(already)} train.tsv) — skipping download.')
else:
    print('Downloading MIntRec 2.0 folder (this is large)...')
    gdown.download_folder(GDRIVE_FOLDER, output=str(MINTREC_DATA), quiet=False, use_cookies=False)

# Unzip any archives that came down (skip if their target dir already exists).
for zp in sorted(MINTREC_DATA.rglob('*.zip')):
    target = zp.with_suffix('')
    if target.exists():
        continue
    print(f'Unzipping {zp.name} ...')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(zp.parent)
print('Done.')

In [ ]:
# Show what landed on disk (top 2 levels) so paths below can be sanity-checked.
for p in sorted(MINTREC_DATA.rglob('*')):
    rel = p.relative_to(MINTREC_DATA)
    if len(rel.parts) <= 2 and (p.is_dir() or p.suffix in {'.tsv', '.csv', '.pkl', '.zip'}):
        tag = '/' if p.is_dir() else ''
        print(f'  {rel}{tag}')

## 3. Load Annotations & Map Raw Videos

The 30 in-scope intent labels (verbatim from the official benchmark config). Annotation TSVs live under an `in-scope/` split with columns `dialogue_id, utterance_id, text, label, ...`; the per-utterance index is `dia{dialogue_id}_utt{utterance_id}`, matching the raw `.mp4` filenames under `raw_data/`.

In [ ]:
INTENT_LABELS = [
    'Acknowledge', 'Advise', 'Agree', 'Apologise', 'Arrange',
    'Ask for help', 'Asking for opinions', 'Care', 'Comfort', 'Complain',
    'Confirm', 'Criticize', 'Doubt', 'Emphasize', 'Explain',
    'Flaunt', 'Greet', 'Inform', 'Introduce', 'Invite',
    'Joke', 'Leave', 'Oppose', 'Plan', 'Praise',
    'Prevent', 'Refuse', 'Taunt', 'Thank', 'Warn',
]
label2id = {l: i for i, l in enumerate(INTENT_LABELS)}
id2label = {i: l for l, i in label2id.items()}
print(f'{len(INTENT_LABELS)} intent classes')

# Locate the in-scope split (folder may be named 'in-scope' or similar).
train_tsvs = list(MINTREC_DATA.rglob('train.tsv'))
assert train_tsvs, 'No train.tsv found — check the download/unzip step.'
# Prefer an in-scope path if multiple matched (in- vs out-of-scope).
train_tsv = next((p for p in train_tsvs if 'in-scope' in str(p).lower().replace('out-of-scope', '')), train_tsvs[0])
SPLIT_DIR = train_tsv.parent
print(f'Split dir : {SPLIT_DIR}')

In [ ]:
def load_split(name):
    """Read one MIntRec2.0 split TSV. Columns: dialogue_id, utterance_id, text, label, ...
    Returns a DataFrame with a derived `index` = dia{d}_utt{u}."""
    df = pd.read_csv(SPLIT_DIR / f'{name}.tsv', sep='\t', dtype=str, keep_default_na=False)
    df.columns = [c.strip() for c in df.columns]
    # Positional fallback — official loader uses col 0,1 (ids), 2 (text), 3 (label).
    cols = list(df.columns)
    dia_c, utt_c, text_c, label_c = cols[0], cols[1], cols[2], cols[3]
    df = df.rename(columns={dia_c: 'dia', utt_c: 'utt', text_c: 'text', label_c: 'label'})
    df['index'] = 'dia' + df['dia'].astype(str) + '_utt' + df['utt'].astype(str)
    return df

train_df = load_split('train')
dev_df = load_split('dev')
test_df = load_split('test')
print(f'Columns   : {list(train_df.columns)}')
print(f'Train: {len(train_df):,}  |  Dev: {len(dev_df):,}  |  Test: {len(test_df):,}')
print(f'Unique labels in train: {train_df["label"].nunique()}')
train_df.head(4)

In [ ]:
# Build index -> raw .mp4 path map by globbing all videos and keying on filename stem.
mp4_paths = list(MINTREC_DATA.rglob('*.mp4'))
stem2path = {p.stem: p for p in mp4_paths}
print(f'Found {len(mp4_paths):,} .mp4 files')

have = train_df['index'].isin(stem2path).mean()
print(f'Train indices with a matching .mp4: {have*100:.1f}%')
if mp4_paths:
    print(f'Example video path: {mp4_paths[0]}')
    print(f'Example stem      : {mp4_paths[0].stem}')

## 4. Inspect Examples

Peek at a few in-scope utterances spanning different intents: print the transcript / label / speaker, and play the extracted audio track. The raw `.mp4` (video + audio) is what we feed the teacher in §6.

In [ ]:
import librosa

# One example per intent, up to 6, that actually has a video on disk.
shown, examples = set(), []
for _, row in train_df.iterrows():
    if row['label'] in shown or row['index'] not in stem2path:
        continue
    examples.append(row)
    shown.add(row['label'])
    if len(examples) == 6:
        break

for i, ex in enumerate(examples):
    vpath = stem2path[ex['index']]
    wav, sr = librosa.load(str(vpath), sr=16000, mono=True)  # decodes mp4 audio via ffmpeg
    speaker = ex.get('speaker', ex.get('Speaker', '?'))
    print(f'\n── Example {i+1} ' + '─' * 30)
    print(f'  Index    : {ex["index"]}')
    print(f'  Intent   : {ex["label"]}')
    print(f'  Speaker  : {speaker}')
    print(f'  Text     : {ex["text"]}')
    print(f'  Duration : {len(wav)/sr:.2f}s  |  Video: {vpath.name}')
    display(IPyAudio(wav, rate=sr))

## 5. Load Qwen2.5-Omni-3B (4-bit quantised — local smoke test)

4-bit NF4 (bitsandbytes) keeps VRAM low enough for a 6 GB 3060. Video tokens are heavier than audio, so use short clips here. On RunPod, swap `quantization_config` for `torch_dtype=torch.float16` to run the teacher at full precision — hidden-state extraction is identical either way.

In [ ]:
from transformers import (
    Qwen2_5OmniForConditionalGeneration,
    Qwen2_5OmniProcessor,
    BitsAndBytesConfig,
)

MODEL_NAME = 'Qwen/Qwen2.5-Omni-3B'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_NAME)
print('Processor loaded.')

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,   # RunPod: replace with torch_dtype=torch.float16
    device_map='auto',
    attn_implementation='eager',
)
model.eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e9
vram_used = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'Model loaded : {MODEL_NAME}  (4-bit NF4)')
print(f'Parameters   : {total_params:.2f}B')
print(f'VRAM used    : {vram_used:.2f} GB')

## 6. Basic Inference Test (video + audio + text → intent)

Feed the raw clip (frames **and** its audio track) plus the transcript, and ask the teacher to pick an intent. `process_mm_info(..., use_audio_in_video=True)` extracts the audio from the same `.mp4` so vision and speech stay aligned.

In [ ]:
from qwen_omni_utils import process_mm_info

sample = examples[0]
vpath = stem2path[sample['index']]

TASK_PROMPT = (
    'You are classifying the speaker\'s intent in a short TV-show clip. '
    'Use the video, the audio, and the transcript. '
    f'Transcript: "{sample["text"]}". '
    'Choose the single best intent from this list and answer with just that label:\n'
    + ', '.join(INTENT_LABELS)
)

conversation = [
    {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful multimodal assistant.'}]},
    {'role': 'user', 'content': [
        {'type': 'video', 'video': str(vpath)},
        {'type': 'text', 'text': TASK_PROMPT},
    ]},
]

text_input = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=True)
inputs = processor(
    text=text_input, audio=audios, images=images, videos=videos,
    return_tensors='pt', padding=True, use_audio_in_video=True,
).to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=32, return_audio=False, use_audio_in_video=True)

prompt_len = inputs['input_ids'].shape[1]
response = processor.batch_decode(generated_ids[:, prompt_len:], skip_special_tokens=True)[0]
print(f'Ground-truth intent : {sample["label"]}')
print(f'Model response      : {response.strip()}')

## 7. Hidden State Extraction (Sanity Check)

These tri-modal hidden states are the distillation target for the tiny student. As in the FSC notebook, the LM backbone lives in `model.thinker` (`generate()` routes through it the same way).

In [ ]:
with torch.no_grad():
    outputs = model.thinker(**inputs, output_hidden_states=True, return_dict=True)

hidden_states = outputs.hidden_states
print(f'Hidden state layers : {len(hidden_states)}  (embedding + {len(hidden_states)-1} transformer blocks)')
print(f'Shape per layer     : {tuple(hidden_states[0].shape)}  [batch, seq_len, hidden_dim]')

mid_idx = len(hidden_states) // 2
mid_repr = hidden_states[mid_idx].float().mean(dim=1)
print(f'\nMid-layer ({mid_idx}) mean-pooled shape : {tuple(mid_repr.shape)}')
vram_peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f'Peak VRAM (this session) : {vram_peak:.2f} GB')